# Fase 1: Análisis Exploratorio y Calidad de Datos (EDA)
Este notebook actúa exclusivamente como capa de consumo interactiva. Toda la lógica dura de inferencia de esquemas, limpieza e imputación de variables se ha encapsulado correctamente en el módulo `src/` aplicando una Arquitectura Limpia.

In [ ]:
import os
import sys

# 1. Agregamos forzosamente la raíz del proyecto al scope para importar 'src' de manera global
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

# 2. Habilitamos la recarga automática de módulos ante cualquier modificación en /src (Hot-Reloading)
%load_ext autoreload
%autoreload 2

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 3. Consumo Desacoplado de la Arquitectura Core
from src.config import settings
from src.ingesta import DataIngestor
from src.procesamiento import DataCleaner

# Configuramos la estética visual para plots financieros
sns.set_theme(style="whitegrid", palette="muted")

## 1.1 Ingesta Restrictiva (Lectura DDL-Agnóstica)
Llamamos al Ingestor para leer el `clientes.csv`. Gracias al soporte de `dtype_schema`, emulamos la rigurosidad impuesta por `schema_postgresql.sql` y delegamos el parseo automático a `datetime64[ns]`.

In [ ]:
# Instanciamos el módulo
ingestor = DataIngestor(raw_dir=settings.DATA_RAW_DIR)

# Esquemas mapeados derivados del schema_postgresql.sql
dtype_clientes = {
    'cliente_id': str,
    'departamento': str,
    'edad': 'Int64',           # Tolera nulos numéricos (pandas 1.0+)
    'ingreso_mensual_estimado': float,
    'score_externo': float,
    'tiene_producto_ahorro': bool
}

try:
    df_clientes_raw = ingestor.read_csv(
        filename=settings.CLIENTES_FILE, 
        dtype_schema=dtype_clientes,
        date_cols=['fecha_registro']
    )
    display(df_clientes_raw.head())
except FileNotFoundError:
    print(f"⚠️ Asegúrate de ubicar 'clientes.csv' en {settings.DATA_RAW_DIR}")

## 1.2 Transformación y Calidad (Data Cleaning)
Empleamos nuestra clase `DataCleaner` para purgar nulos, auditar outliers e inyectar completitud. Observa que el notebook **no posee lógica de pandas `.fillna()`** en celdas sueltas, combatiendo directamente el antipatrón de "Código Espagueti".

In [ ]:
cleaner = DataCleaner()

if 'df_clientes_raw' in locals():
    # El método 'clean_clientes' se encarga internamente de imputar ingresos por mediana
    # y aplicar el umbral estricto para outliers de edad (>110, <18).
    df_clientes_clean = cleaner.clean_clientes(df_clientes_raw)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.histplot(df_clientes_clean['edad'], bins=30, kde=True, ax=axes[0], color='teal')
    axes[0].set_title('Distribución de Edad (Post-Limpieza)')
    
    sns.histplot(df_clientes_clean['ingreso_mensual_estimado'], bins=50, kde=True, ax=axes[1], color='coral')
    axes[1].set_title('Distribución de Ingresos (Imputados)')
    plt.show()